In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize Messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="gpt-4o-mini",
        trigger=("tokens", 100),
        keep=("messages",1)
    )],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage

from pprint import pprint

response = agent.invoke(
    {
        "messages":[
            HumanMessage(content="What is the captail of the Moon?"),
            AIMessage(content="The capital of the Moon is called Lunapolis"),
            HumanMessage(content="What is the weather in Lunapolis?"),
            AIMessage(content="Sky is clear with a high of 120C and a low of -100C."),
            HumanMessage(content="How many spice miners live in Lunapolis?"),
            AIMessage(content="There are 100,000 spice miners living in Lunapolis"),
            HumanMessage(content="Do you think the spice miners union will strike?"),
            AIMessage(content="The union is considering a strike due to low wages and poor working conditions"),
            HumanMessage(content="If you were Lunapolis' new president, how would you responsd to the spice miners union?"),
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(response)


{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, the capital of the Moon, including its weather, population of spice miners, and possible union actions.\n\n## SUMMARY\n- The capital of the Moon is Lunapolis.\n- The weather in Lunapolis is clear, with a high of 120°C and a low of -100°C.\n- There are 100,000 spice miners living in Lunapolis.\n- The spice miners' union is considering a strike due to low wages and poor working conditions.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='d013a3a8-60cd-457d-b91e-26d6575f0136'),
              HumanMessage(content="If you were Lunapolis' new president, how would you responsd to the spice miners union?", additional_kwargs={}, response_metadata={}, id='b4d7d68a-e451-44ee-ad2e-9b4fb3e76cb2'),
              AIMessage(content='If I were Lunapolis’ new president, 

### Summarizing Messages

- Here we were able to conclude the older conversations, store the last message where we asked how would you respond to the spice miners union by using the summarization middleware and then getting the AIMessage as a response to the last question we had asked.

In [10]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, the capital of the Moon, including its weather, population of spice miners, and possible union actions.

## SUMMARY
- The capital of the Moon is Lunapolis.
- The weather in Lunapolis is clear, with a high of 120°C and a low of -100°C.
- There are 100,000 spice miners living in Lunapolis.
- The spice miners' union is considering a strike due to low wages and poor working conditions.

## ARTIFACTS
None

## NEXT STEPS
None


# Trim/Delete Messages

- What we should do in case we do not want to summarize our messages, rather we wanted to trim, delete or remove the messages entirely, or what we should do if we want granular control over what messages get deleted rather than just deleting the oldest messages. This can be achieved using the custom middleware function, which we can build ourselves.

- To do this we have before and after agent decorators around functions and we turn them into middleware which runs once per run at the very start or at the very end. You can also insert custom middleware before and after every model call using the before model and after model decorators, a function could run several times per run due to this.

![](https://i.imgur.com/RAQk92K.png)

- Below we see how we can remove all previous ToolMessages from the state before the start of an agent run. We can do this in the following manner:

In [29]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent, after_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages_before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [18]:
agent_before = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages_before_agent],
)

In [19]:
response = agent_before.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(content="blorp-x7 initating diagnostic ping...", tool_call_id="1"),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on"),
            ToolMessage(content="temp=42C voltage=2.9V ... greeble complete.", tool_call_id="2"),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?")
        ]
    },
    {"configurable": {"thread_id":"2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='6d8165f3-f021-49a9-9213-8b849d3c9d48'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='0b4a9b1c-d7a7-47ec-b78a-44a6b3b83d44', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on", additional_kwargs={}, response_metadata={}, id='acaf611b-9450-4324-936b-5340fdcaf6cf'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='8187ac3b-d1a3-4429-a02f-d7eb84ebf33c', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='1597e876-3c73-4593-aeb1-d662b21de1db'),
              AIMessage(content='I can’t measure its temperature from here, but I can help you c

In [20]:
print(response["messages"][-1].content) # Here we use the before_agent as middleware so Agent would not know the temperature which was in the tool call.

I can’t measure its temperature from here, but I can help you check and troubleshoot it. To tailor steps, tell me what kind of device this is (e.g., PC/laptop, Android phone, iPhone, tablet, game console, router, etc.) and the model if you know it.

In the meantime, here are safe, general steps if a device feels hot or won’t power on:

- If it’s hot to the touch or smells burnt:
  - Power it off immediately and unplug from power.
  - Let it cool in a well-ventilated area for at least 15–30 minutes before trying again.
  - Don’t use or charge it while it’s hot.

- Check the power supply:
  - Try a different outlet.
  - If you have a detachable charger/brick, try a different one that’s compatible.
  - Inspect the power cable for damage.

- Remove accessories and give a reset a try:
  - For laptops/PCs: hold the power button for 10–15 seconds to force shutdown, then reconnect power and try to start again.
  - For phones: try a forced restart (often Power + Volume Down/Up, varies by model)

# After agent call demo

In [28]:
@after_agent
def trim_messages_after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state after the agent call"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent_after = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages_after_agent],
)

**Root cause**: Every ToolMessage in OpenAI's API must follow an AIMessage that contains a tool_calls entry with a matching id. The original demo skipped these, which is fine for before_agent (those messages get stripped before the
   model call) but breaks after_agent (the model sees them first).

- The fix: Add a "gateway" AIMessage with tool_calls before each ToolMessage:

- Before:  HumanMessage → ToolMessage ← INVALID (no preceding tool_calls)

- After:   HumanMessage
        AIMessage(tool_calls=[{id: "1", name: "run_diagnostic"}])
        ToolMessage(tool_call_id="1")  ← valid, matched by id

What you'll now see in the verification cell:
- The agent response mentions 42C — proving it had access to the ToolMessage content during its run
- ToolMessages remaining in state: 0 — proving after_agent cleaned them up afterward

In [ ]:
# For after_agent to work, ToolMessages must be preceded by an AIMessage
# with a matching tool_calls entry — otherwise OpenAI rejects the request
# before the agent even runs (and after_agent never gets a chance to clean up).
#
# The before_agent demo could skip this because it strips ToolMessages first.
# Here we construct a valid message sequence so the model call succeeds,
# then after_agent removes the ToolMessages from state.

response = agent_after.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            AIMessage(
                content="",
                tool_calls=[{"id": "1", "name": "run_diagnostic", "args": {}}]
            ),
            ToolMessage(content="blorp-x7 initiating diagnostic ping...", tool_call_id="1"),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on"),
            AIMessage(
                content="",
                tool_calls=[{"id": "2", "name": "get_diagnostics", "args": {}}]
            ),
            ToolMessage(content="temp=42C voltage=2.9V ... greeble complete.", tool_call_id="2"),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?")
        ]
    },
    {"configurable": {"thread_id": "4"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='e33c86a0-2e3d-4435-9695-4d8e99cd6dc5'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='9ebfe327-6b3d-4c52-bcb4-a6ba597681da', tool_calls=[{'name': 'run_diagnostic', 'args': {}, 'id': '1', 'type': 'tool_call'}], invalid_tool_calls=[]),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='d92c6224-fafa-4454-b2fc-7590e85690be', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on", additional_kwargs={}, response_metadata={}, id='761c1c7e-d5ca-4fb0-a0d5-2cc7ccab5b04'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='0a5daccd-a971-479b-8ea9-1ab4563f5dbb', tool_calls=[{'name': 'get_diagnostics', 'args': {}, 'id': '2', 'type': 'tool_call'}], invalid_tool_calls=[]),
    

In [31]:

# The agent's response should mention 42C — it saw the ToolMessage during its run.
print("=== Agent response (should mention 42C) ===")
print(response["messages"][-1].content)

# after_agent middleware should have stripped all ToolMessages from state.
from langchain.messages import ToolMessage as TM
tool_msgs_remaining = [m for m in response["messages"] if isinstance(m, TM)]
print(f"\n=== ToolMessages remaining in state: {len(tool_msgs_remaining)} (should be 0) ===")


=== Agent response (should mention 42C) ===
The device temperature is 42°C.

Notes:
- 42°C is generally within a normal operating range. If it starts climbing toward 50–60°C, or if it won’t power on after cooling, try these quick steps:
  - Make sure vents are clear and the device isn’t sitting on a soft surface.
  - Place it on a hard, flat surface in a cooler room and give it a few minutes to cool.
  - If it’s safe, perform a power cycle: unplug, press and hold the power button for 10–15 seconds, then reconnect and try powering on.

If it still won’t turn on after cooling, tell me and I’ll guide you through further checks or a diagnostic run.

=== ToolMessages remaining in state: 0 (should be 0) ===
